In [ ]:
import gc
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from collections import deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms


# ============================================================
# 0) Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


def mean_deque(d: deque) -> float:
    return float(np.mean(list(d))) if len(d) > 0 else float("nan")


def fmt_table6(df: pd.DataFrame) -> str:
    # match paper-like 4-decimal display
    df2 = df.copy()
    for c in df2.columns:
        if pd.api.types.is_numeric_dtype(df2[c]):
            df2[c] = df2[c].map(lambda v: "–" if pd.isna(v) else f"{v:.4f}")
    return df2.to_string(index=False)


# ============================================================
# 1) Table 6 config (paper notation)
# ============================================================

@dataclass
class Table6Config:
    # FL (small, Colab-friendly; matches E2 style)
    K: int = 12          # number of clients
    R: int = 25          # rounds
    E: int = 1           # local epochs
    B: int = 64          # batch size
    eta: float = 0.01    # learning rate
    mu: float = 0.9      # momentum

    # Non-IID
    alpha_dir: float = 0.5

    # Head/Tail split
    head: Tuple[int, ...] = (0, 1, 2, 3, 4)
    tail: Tuple[int, ...] = (5, 6, 7, 8, 9)

    # Public leak (for head-only gaming)
    phi: float = 1.0

    # Profiles
    gf: float = 0.30         # gaming fraction in "gaming" condition
    benign_frac: float = 0.10  # keep as in your E2 code
    s_update: float = 1.75   # update scaling factor for scaling-type gaming

    # DP-like perturbation: clip + Gaussian noise
    C: float = 1.0
    nu_grid: Tuple[float, ...] = (0.00, 0.05, 0.10)  # ν ∈ {0, 0.05, 0.10}

    # Tail-mean window (paper: last K rounds; here name it L to avoid clash with #clients)
    L: int = 7

    seed: int = 42
    root: str = "./data"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG = Table6Config()


# ============================================================
# 2) Data prep (Fashion-MNIST)
#    - 50k local partitioned
#    - 10k public candidates -> head-only public val (+ leak subset)
#    - per-client heldout tail-eval from local split
# ============================================================

def make_dirichlet_partitions(labels: np.ndarray, K: int, alpha_dir: float, rng: np.random.Generator) -> List[List[int]]:
    n_classes = int(labels.max()) + 1
    client_indices: List[List[int]] = [[] for _ in range(K)]

    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue
        proportions = rng.dirichlet(alpha_dir * np.ones(K))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shards = np.split(idx_k, splits[:-1])
        for cid, shard in enumerate(shards):
            client_indices[cid].extend(shard.tolist())

    for cid in range(K):
        rng.shuffle(client_indices[cid])

    return client_indices


def filter_subset_by_labels(
    base_subset: Subset,
    train_local_set: Subset,
    full_train: datasets.FashionMNIST,
    allowed: Tuple[int, ...],
) -> Subset:
    allowed_set = set(allowed)
    kept = []
    for idx_in_local in base_subset.indices:
        global_idx = train_local_set.indices[idx_in_local]
        y = int(full_train.targets[global_idx])
        if y in allowed_set:
            kept.append(idx_in_local)
    return Subset(train_local_set, kept)


def split_subset_train_eval(base_subset: Subset, seed: int, eval_ratio: float = 0.2) -> Tuple[Subset, Subset]:
    idxs = list(base_subset.indices)
    rng = np.random.default_rng(seed)
    rng.shuffle(idxs)
    n_eval = int(round(eval_ratio * len(idxs)))
    ev = idxs[:n_eval]
    tr = idxs[n_eval:]
    return Subset(base_subset.dataset, tr), Subset(base_subset.dataset, ev)


def oversample_tail(
    train_subset: Subset,
    train_local_set: Subset,
    full_train: datasets.FashionMNIST,
    tail: Tuple[int, ...],
    factor: int = 2,
) -> ConcatDataset:
    tail_sub = filter_subset_by_labels(train_subset, train_local_set, full_train, tail)
    if len(tail_sub) == 0:
        return ConcatDataset([train_subset])
    return ConcatDataset([train_subset] + [tail_sub for _ in range(factor)])


def prepare_data(cfg: Table6Config):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    full_train = datasets.FashionMNIST(root=cfg.root, train=True, download=True, transform=transform)

    # 60k -> 50k local + 10k public candidates
    n_local = 50_000
    idx_all = np.arange(len(full_train))
    idx_local = idx_all[:n_local]
    idx_public = idx_all[n_local:]

    train_local_set = Subset(full_train, idx_local.tolist())
    targets_all = np.array(full_train.targets)

    # public head val
    head_mask = np.isin(targets_all, np.array(cfg.head, dtype=int))
    public_head_idx = np.array([i for i in idx_public if head_mask[i]], dtype=int)
    rng = np.random.default_rng(cfg.seed)
    rng.shuffle(public_head_idx)

    leak_size = int(cfg.phi * len(public_head_idx))
    leak_idx = public_head_idx[:leak_size]

    public_val_set = Subset(full_train, public_head_idx.tolist())
    public_val_loader = DataLoader(
        public_val_set,
        batch_size=cfg.B,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )
    leak_subset = Subset(full_train, leak_idx.tolist())

    # dirichlet partitions on local 50k
    train_targets_local = np.array(full_train.targets)[idx_local]
    client_indices = make_dirichlet_partitions(
        labels=train_targets_local,
        K=cfg.K,
        alpha_dir=cfg.alpha_dir,
        rng=np.random.default_rng(cfg.seed)
    )
    client_base = [Subset(train_local_set, idxs) for idxs in client_indices]

    # per-client train + tail-eval loaders
    client_train: List[Subset] = []
    client_tail_eval_loaders: List[Optional[DataLoader]] = []

    for cid in range(cfg.K):
        tr, ev = split_subset_train_eval(client_base[cid], seed=cfg.seed + 123 + cid, eval_ratio=0.2)
        client_train.append(tr)

        tail_ev = filter_subset_by_labels(ev, train_local_set, full_train, cfg.tail)
        if len(tail_ev) == 0:
            client_tail_eval_loaders.append(None)
        else:
            client_tail_eval_loaders.append(
                DataLoader(
                    tail_ev,
                    batch_size=cfg.B,
                    shuffle=False,
                    num_workers=0,
                    pin_memory=torch.cuda.is_available()
                )
            )

    cleanup()
    return full_train, train_local_set, client_train, client_tail_eval_loaders, public_val_loader, leak_subset


# ============================================================
# 3) Model + local training
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def make_model(device: torch.device) -> nn.Module:
    return SimpleCNN().to(device)


def local_train(model: nn.Module, dataset, cfg: Table6Config) -> None:
    loader = DataLoader(
        dataset,
        batch_size=cfg.B,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )
    opt = optim.SGD(model.parameters(), lr=cfg.eta, momentum=cfg.mu)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(cfg.E):
        for x, y in loader:
            x = x.to(cfg.device, non_blocking=True)
            y = y.to(cfg.device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            opt.step()


# ============================================================
# 4) Strategies (aligned vs gaming)
# ============================================================

STR_HONEST = "HONEST"
STR_BENIGN = "BENIGN_TAIL_UPWEIGHT"
STR_GAME_HEAD = "GAMING_HEAD_ONLY_LEAK"
STR_GAME_SCALE = "GAMING_UPDATE_SCALING"


def assign_strategies(cfg: Table6Config, gaming_frac: float, benign_frac: float, rng: np.random.Generator) -> Dict[int, str]:
    ids = np.arange(cfg.K)
    rng.shuffle(ids)

    n_benign = int(round(benign_frac * cfg.K))
    n_gaming = int(round(gaming_frac * cfg.K))

    benign_ids = ids[:n_benign]
    gaming_ids = ids[n_benign:n_benign + n_gaming]
    honest_ids = ids[n_benign + n_gaming:]

    strat = {int(cid): STR_HONEST for cid in honest_ids}
    for cid in benign_ids:
        strat[int(cid)] = STR_BENIGN

    # split gaming ids into head-only and update-scaling
    half = len(gaming_ids) // 2
    for cid in gaming_ids[:half]:
        strat[int(cid)] = STR_GAME_HEAD
    for cid in gaming_ids[half:]:
        strat[int(cid)] = STR_GAME_SCALE

    return strat


def build_client_train_dataset(
    cid: int,
    strat: str,
    cfg: Table6Config,
    base_train: Subset,
    full_train,
    train_local_set,
    leak_subset,
):
    if strat == STR_HONEST:
        return base_train

    if strat == STR_BENIGN:
        return oversample_tail(base_train, train_local_set, full_train, cfg.tail, factor=2)

    if strat == STR_GAME_HEAD:
        head_only = filter_subset_by_labels(base_train, train_local_set, full_train, cfg.head)
        return ConcatDataset([head_only, leak_subset])

    if strat == STR_GAME_SCALE:
        return base_train

    raise ValueError(strat)


# ============================================================
# 5) DP-like perturbation on transmitted update: clip + Gaussian noise
#     delta := scale*(local - global)
#     delta <- clip(delta, C) + N(0, (ν*C)^2 I)
# ============================================================

def _flatten_sd(sd: Dict[str, torch.Tensor]) -> torch.Tensor:
    return torch.cat([sd[k].reshape(-1) for k in sd.keys()], dim=0)


def _unflatten_to_sd(vec: torch.Tensor, template: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    off = 0
    for k in template.keys():
        n = template[k].numel()
        out[k] = vec[off:off + n].reshape(template[k].shape)
        off += n
    return out


def dp_perturb_delta(delta_cpu: Dict[str, torch.Tensor], C: float, nu: float) -> Dict[str, torch.Tensor]:
    v = _flatten_sd(delta_cpu)

    # L2 clip
    norm = torch.norm(v, p=2)
    if norm > C:
        v = v * (C / (norm + 1e-12))

    # Gaussian noise
    if nu > 0:
        std = nu * C
        v = v + torch.randn_like(v) * std

    return _unflatten_to_sd(v, delta_cpu)


# ============================================================
# 6) One run: returns tail-mean (last L rounds) of M and W
#     - M: public head metric (accuracy on public head val)
#     - W: tail welfare = mean over per-client tail-eval accuracies (nanmean)
# ============================================================

def run_once(
    cfg: Table6Config,
    data_bundle,
    gaming_frac: float,
    benign_frac: float,
    nu: float,
    seed_offset: int,
) -> Tuple[float, float]:
    full_train, train_local_set, client_train, client_tail_eval_loaders, public_val_loader, leak_subset = data_bundle

    # per-run determinism
    set_seed(cfg.seed + seed_offset)
    rng = np.random.default_rng(cfg.seed + seed_offset)

    strat_map = assign_strategies(cfg, gaming_frac=gaming_frac, benign_frac=benign_frac, rng=rng)

    # build per-client training datasets
    client_train_ds = []
    for cid in range(cfg.K):
        ds = build_client_train_dataset(
            cid=cid,
            strat=strat_map[cid],
            cfg=cfg,
            base_train=client_train[cid],
            full_train=full_train,
            train_local_set=train_local_set,
            leak_subset=leak_subset,
        )
        client_train_ds.append(ds)

    # global state on CPU
    model0 = make_model(cfg.device)
    global_sd_cpu = {k: v.detach().to("cpu").clone() for k, v in model0.state_dict().items()}
    del model0
    cleanup()

    M_last = deque(maxlen=cfg.L)
    W_last = deque(maxlen=cfg.L)

    for _ in range(cfg.R):
        # streaming FedAvg: accumulate perturbed deltas (not full state list)
        sum_delta = {k: torch.zeros_like(v, device="cpu") for k, v in global_sd_cpu.items()}

        for cid in range(cfg.K):
            # local train from global
            local_model = make_model(cfg.device)
            local_model.load_state_dict(global_sd_cpu, strict=True)

            local_train(local_model, client_train_ds[cid], cfg)
            local_sd = local_model.state_dict()

            # scale factor for update-scaling gamers
            scale = cfg.s_update if (strat_map[cid] == STR_GAME_SCALE and gaming_frac > 0) else 1.0

            # delta on CPU: scale*(local - global)
            delta_cpu = {}
            for k in global_sd_cpu.keys():
                delta_cpu[k] = scale * (local_sd[k].detach().to("cpu") - global_sd_cpu[k])

            # DP-like clip+noise on delta
            delta_cpu = dp_perturb_delta(delta_cpu, C=cfg.C, nu=nu)

            # accumulate
            with torch.no_grad():
                for k in sum_delta.keys():
                    sum_delta[k] += delta_cpu[k]

            del local_model, local_sd, delta_cpu
            cleanup()

        # update global: global + avg(delta)
        with torch.no_grad():
            for k in global_sd_cpu.keys():
                global_sd_cpu[k] = global_sd_cpu[k] + (sum_delta[k] / float(cfg.K))

        del sum_delta
        cleanup()

        # evaluate M and W
        eval_model = make_model(cfg.device)
        eval_model.load_state_dict(global_sd_cpu, strict=True)

        M_t = accuracy(eval_model, public_val_loader, cfg.device)

        tail_accs = []
        for cid in range(cfg.K):
            loader = client_tail_eval_loaders[cid]
            if loader is None:
                tail_accs.append(np.nan)
            else:
                tail_accs.append(accuracy(eval_model, loader, cfg.device))
        W_t = float(np.nanmean(np.array(tail_accs, dtype=float)))

        M_last.append(M_t)
        W_last.append(W_t)

        del eval_model
        cleanup()

    return mean_deque(M_last), mean_deque(W_last)


# ============================================================
# 7) Table 6 driver: print ONLY Table 6
# ============================================================

def run_table_6(cfg: Table6Config) -> pd.DataFrame:
    # fixed dataset/partitions
    set_seed(cfg.seed)
    data_bundle = prepare_data(cfg)

    # reference welfare W_ref from aligned, ν=0 run (separate fixed run)
    M_ref, W_ref = run_once(
        cfg,
        data_bundle,
        gaming_frac=0.0,
        benign_frac=cfg.benign_frac,
        nu=0.0,
        seed_offset=0,
    )

    rows = []
    seed_offset = 10

    for nu in cfg.nu_grid:
        # aligned at this ν
        M_a, W_a = run_once(
            cfg,
            data_bundle,
            gaming_frac=0.0,
            benign_frac=cfg.benign_frac,
            nu=float(nu),
            seed_offset=seed_offset,
        )
        seed_offset += 1

        # gaming at this ν (gf=0.30)
        M_g, W_g = run_once(
            cfg,
            data_bundle,
            gaming_frac=cfg.gf,
            benign_frac=cfg.benign_frac,
            nu=float(nu),
            seed_offset=seed_offset,
        )
        seed_offset += 1

        dW = W_a - W_g
        dgap = (M_g - W_g) - (M_a - W_a)
        pog_ref_g = np.nan if (not np.isfinite(W_ref) or W_ref <= 1e-12) else (W_ref - W_g) / W_ref

        rows.append({
            "nu": float(nu),
            "M_aligned": float(M_a),
            "W_aligned": float(W_a),
            "M_gaming": float(M_g),
            "W_gaming": float(W_g),
            "Delta_W": float(dW),
            "Delta_gap": float(dgap),
            "PoG(ref)_gaming": float(pog_ref_g),
        })

    # cleanup big bundle
    del data_bundle
    cleanup()

    df = pd.DataFrame(rows).sort_values("nu").reset_index(drop=True)
    return df


if __name__ == "__main__":
    df6 = run_table_6(CFG)

    # Print ONLY Table 6 (no extra logs)
    print("Table 6: Noise/privacy trade-off under DP-like update perturbations.")
    print(fmt_table6(df6))